In [1]:
import pandas as pd
import numpy as np

In [2]:
dataset_path='PJME_hourly.csv'

In [5]:
df=pd.read_csv(dataset_path,parse_dates=['Datetime'],index_col='Datetime')
df = df.sort_index()
df = df[~df.index.duplicated(keep='first')]

In [8]:
def create_time_features(df):
    df = df.copy()
    df['hour']=df.index.hour
    df['dayofweek']=df.index.dayofweek
    df['quarter']=df.index.quarter
    df['month']=df.index.month
    df['year']=df.index.year
    df['dayofyear'] = df.index.dayofyear
    df['is_weekend'] = df['dayofweek'].isin([5,6]).astype(int)
    return df

In [9]:
df = create_time_features(df)

In [10]:
df

,PJME_MW,hour,dayofweek,quarter,month,year,dayofyear,is_weekend
Datetime,,,,,,,,
2002-01-01 01:00:00,30393.0,1,1,1,1,2002,1,0
2002-01-01 02:00:00,29265.0,2,1,1,1,2002,1,0
2002-01-01 03:00:00,28357.0,3,1,1,1,2002,1,0
2002-01-01 04:00:00,27899.0,4,1,1,1,2002,1,0
2002-01-01 05:00:00,28057.0,5,1,1,1,2002,1,0
...,...,...,...,...,...,...,...,...
2018-08-02 20:00:00,44057.0,20,3,3,8,2018,214,0
2018-08-02 21:00:00,43256.0,21,3,3,8,2018,214,0
2018-08-02 22:00:00,41552.0,22,3,3,8,2018,214,0


In [11]:
#3 Chronological Split (Cutoff at Jan 1, 2015)
split_date='2015-01-01'

In [12]:
train = df.loc[df.index<split_date].copy()
test = df.loc[df.index>=split_date].copy()

In [13]:
print(f"Training set rows: {len(train)} (Dates: {train.index.min()} to {train.index.max()})")
print(f"Testing set rows: {len(test)} (Dates: {test.index.min()} to {test.index.max()})")

Training set rows: 113925 (Dates: 2002-01-01 01:00:00 to 2014-12-31 23:00:00)
Testing set rows: 31437 (Dates: 2015-01-01 00:00:00 to 2018-08-03 00:00:00)


In [18]:
def create_lag_features(df):
    df = df.copy()

    # Map target value back in time
    df['lag_1_hour']=df['PJME_MW'].shift(1)
    df['lag_24_hours']=df['PJME_MW'].shift(24)
    df['lag_7_days']=df['PJME_MW'].shift(7*24)
    return df
df=create_lag_features(df)

print("--- Early Rows with NaNs due to shifting ---")
print(df[['PJME_MW', 'lag_1_hour', 'lag_24_hours', 'lag_7_days']].head(5))

print("\n--- Rows after 168 hours (7 days) where lag values fill up ---")
print(df[['PJME_MW', 'lag_1_hour', 'lag_24_hours', 'lag_7_days']].iloc[170:175])


--- Early Rows with NaNs due to shifting ---
                     PJME_MW  lag_1_hour  lag_24_hours  lag_7_days
Datetime                                                          
2002-01-01 01:00:00  30393.0         NaN           NaN         NaN
2002-01-01 02:00:00  29265.0     30393.0           NaN         NaN
2002-01-01 03:00:00  28357.0     29265.0           NaN         NaN
2002-01-01 04:00:00  27899.0     28357.0           NaN         NaN
2002-01-01 05:00:00  28057.0     27899.0           NaN         NaN

--- Rows after 168 hours (7 days) where lag values fill up ---
                     PJME_MW  lag_1_hour  lag_24_hours  lag_7_days
Datetime                                                          
2002-01-08 03:00:00  28375.0     28670.0       25641.0     28357.0
2002-01-08 04:00:00  28542.0     28375.0       25666.0     27899.0
2002-01-08 05:00:00  29261.0     28542.0       26328.0     28057.0
2002-01-08 06:00:00  31348.0     29261.0       28267.0     28654.0
2002-01-08 07:00:00 

In [19]:
df

,PJME_MW,hour,dayofweek,quarter,month,year,dayofyear,is_weekend,lag_1_hour,lag_24_hours,lag_7_days
Datetime,,,,,,,,,,,
2002-01-01 01:00:00,30393.0,1,1,1,1,2002,1,0,NaN,NaN,NaN
2002-01-01 02:00:00,29265.0,2,1,1,1,2002,1,0,30393.0,NaN,NaN
2002-01-01 03:00:00,28357.0,3,1,1,1,2002,1,0,29265.0,NaN,NaN
2002-01-01 04:00:00,27899.0,4,1,1,1,2002,1,0,28357.0,NaN,NaN
2002-01-01 05:00:00,28057.0,5,1,1,1,2002,1,0,27899.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2018-08-02 20:00:00,44057.0,20,3,3,8,2018,214,0,45641.0,46912.0,46337.0
2018-08-02 21:00:00,43256.0,21,3,3,8,2018,214,0,44057.0,45985.0,44542.0
2018-08-02 22:00:00,41552.0,22,3,3,8,2018,214,0,43256.0,44094.0,42638.0


In [22]:
def create_rolling_features(df):
    df=df.copy()
    df['rolling_mean_24h']= df['PJME_MW'].shift(1).rolling(window=24).mean()
    df['rolling_mean_7d']= df['PJME_MW'].shift(1).rolling(window=24*7).mean()
    return df
df = create_rolling_features(df)
df_clean=df.dropna()
print("--- Final Cleaned Dataset Info ---")
df_clean.info()
print(f"Dropped {len(df)-len(df_clean)} rows containing NaN values")



--- Final Cleaned Dataset Info ---
<class 'pandas.DataFrame'>
DatetimeIndex: 145194 entries, 2002-01-08 01:00:00 to 2018-08-03 00:00:00
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   PJME_MW           145194 non-null  float64
 1   hour              145194 non-null  int32  
 2   dayofweek         145194 non-null  int32  
 3   quarter           145194 non-null  int32  
 4   month             145194 non-null  int32  
 5   year              145194 non-null  int32  
 6   dayofyear         145194 non-null  int32  
 7   is_weekend        145194 non-null  int64  
 8   lag_1_hour        145194 non-null  float64
 9   lag_24_hours      145194 non-null  float64
 10  lag_7_days        145194 non-null  float64
 11  rolling_mean_24h  145194 non-null  float64
 12  rolling_mean_7d   145194 non-null  float64
dtypes: float64(6), int32(6), int64(1)
memory usage: 12.2 MB
Dropped 168 rows containing NaN values


In [23]:
df_clean

,PJME_MW,hour,dayofweek,quarter,month,year,dayofyear,is_weekend,lag_1_hour,lag_24_hours,lag_7_days,rolling_mean_24h,rolling_mean_7d
Datetime,,,,,,,,,,,,,
2002-01-08 01:00:00,29445.0,1,1,1,1,2002,8,0,31187.0,26862.0,30393.0,33452.583333,32519.511905
2002-01-08 02:00:00,28670.0,2,1,1,1,2002,8,0,29445.0,25976.0,29265.0,33560.208333,32513.869048
2002-01-08 03:00:00,28375.0,3,1,1,1,2002,8,0,28670.0,25641.0,28357.0,33672.458333,32510.327381
2002-01-08 04:00:00,28542.0,4,1,1,1,2002,8,0,28375.0,25666.0,27899.0,33786.375000,32510.434524
2002-01-08 05:00:00,29261.0,5,1,1,1,2002,8,0,28542.0,26328.0,28057.0,33906.208333,32514.261905
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-08-02 20:00:00,44057.0,20,3,3,8,2018,214,0,45641.0,46912.0,46337.0,40021.875000,35913.809524
2018-08-02 21:00:00,43256.0,21,3,3,8,2018,214,0,44057.0,45985.0,44542.0,39902.916667,35900.238095
2018-08-02 22:00:00,41552.0,22,3,3,8,2018,214,0,43256.0,44094.0,42638.0,39789.208333,35892.583333
